#  Financial Transaction Anomaly Classifier
**Detecting Fraudulent Payments Using Ensemble & Boosting Methods**


## Section A — Setup & Dependencies

In [ ]:
! pip install kaggle -q

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json
print('Kaggle credentials configured.')

## Section B — Dataset Acquisition


In [ ]:
import kaggle
kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    'rupakroy/online-payments-fraud-detection-dataset',
    path='./',
    unzip=True
)
print('Dataset downloaded and extracted.')

## Section C — Loading & First Look at the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

transactions = pd.read_csv('PS_20174392719_1491204439457_log.csv')
print('Dataset shape:', transactions.shape)
transactions.head()

In [ ]:
print('--- Column Summary ---')
for col in transactions.columns:
    dtype  = transactions[col].dtype
    nuniq  = transactions[col].nunique()
    nulls  = transactions[col].isnull().sum()
    print(f'{col:<22} dtype={str(dtype):<10} unique={nuniq:<10} nulls={nulls}')

In [ ]:
total_txns  = len(transactions)
n_fraud     = transactions['isFraud'].sum()
n_legit     = total_txns - n_fraud
pct_fraud   = n_fraud / total_txns * 100

print(f'Total Transactions  : {total_txns:,}')
print(f'Fraudulent          : {n_fraud:,}  ({pct_fraud:.4f}%)')
print(f'Legitimate          : {n_legit:,}')

## Section D — Exploratory Data Analysis


In [ ]:
# 1 — Class imbalance visualisation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

class_labels = transactions['isFraud'].map({0: 'Legitimate', 1: 'Fraud'})
vc = class_labels.value_counts()
ax1.bar(vc.index, vc.values, color=['#4C72B0', '#DD8452'], edgecolor='white')
ax1.set_title('Class Imbalance — Full Dataset', fontsize=13)
ax1.set_ylabel('Number of Transactions')
for bar_rect, val in zip(ax1.patches, vc.values):
    ax1.text(bar_rect.get_x() + bar_rect.get_width()/2, val + 5000,
             f'{val:,}', ha='center', fontsize=9)

ax2.pie(vc.values, labels=vc.index, autopct='%1.3f%%',
        colors=['#4C72B0', '#DD8452'], startangle=90, wedgeprops={'edgecolor':'white'})
ax2.set_title('Proportion of Fraud', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 2 — Fraud breakdown by transaction type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

volume_by_type = transactions.groupby('type').size().sort_values(ascending=False)
axes[0].barh(volume_by_type.index, volume_by_type.values, color='#4C72B0')
axes[0].set_title('Transaction Volume by Type')
axes[0].set_xlabel('Count')

fraud_pct_by_type = transactions.groupby('type')['isFraud'].mean().sort_values(ascending=False) * 100
bar_cols = ['#DD8452' if v > 0 else '#4C72B0' for v in fraud_pct_by_type.values]
axes[1].bar(fraud_pct_by_type.index, fraud_pct_by_type.values, color=bar_cols)
axes[1].set_title('Fraud Rate (%) per Transaction Type')
axes[1].set_ylabel('Fraud Rate (%)')
plt.tight_layout()
plt.show()
print(fraud_pct_by_type.round(4))

In [ ]:
# 3 — Transaction amount: fraud vs legit (log scale)
fraud_subset  = transactions[transactions['isFraud'] == 1]
legit_subset  = transactions[transactions['isFraud'] == 0]

plt.figure(figsize=(11, 5))
plt.hist(np.log1p(legit_subset['amount']), bins=70, alpha=0.55,
         label='Legitimate', color='#4C72B0', density=True)
plt.hist(np.log1p(fraud_subset['amount']),  bins=70, alpha=0.70,
         label='Fraud',      color='#DD8452', density=True)
plt.xlabel('log(1 + Amount)')
plt.ylabel('Density')
plt.title('Transaction Amount Distribution by Class (Log Scale)')
plt.legend()
plt.show()

In [ ]:
# 4 — Correlation heatmap on sampled data
numeric_preview = transactions[['step','amount','oldbalanceOrg','newbalanceOrig',
                                  'oldbalanceDest','newbalanceDest','isFraud']].sample(5000, random_state=7)
corr_matrix = numeric_preview.corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# 5 — Sender balance before vs after for fraud transactions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for subset, lbl, col in [(fraud_subset, 'Fraud', '#DD8452'), (legit_subset.sample(8000, random_state=7), 'Legit (sample)', '#4C72B0')]:
    axes[0].scatter(np.log1p(subset['oldbalanceOrg']),
                    np.log1p(subset['newbalanceOrig']),
                    alpha=0.3, s=5, label=lbl, color=col)
axes[0].set_xlabel('log(1 + Old Balance Orig)')
axes[0].set_ylabel('log(1 + New Balance Orig)')
axes[0].set_title('Sender Balance Before vs After')
axes[0].legend(markerscale=4)

axes[1].boxplot(
    [np.log1p(fraud_subset['amount']), np.log1p(legit_subset.sample(8000, random_state=7)['amount'])],
    labels=['Fraud', 'Legitimate']
)
axes[1].set_title('Amount Distribution by Class')
axes[1].set_ylabel('log(1 + Amount)')
plt.tight_layout()
plt.show()

## Section E — Feature Engineering



In [ ]:
data = transactions.copy()

# Remove isFlaggedFraud — unreliable system flag, potential leakage
data = data.drop(columns=['isFlaggedFraud'])

# --- New engineered features ---
data['txn_to_balance_ratio'] = data['amount'] / (data['oldbalanceOrg'] + 1)
data['sender_zeroed_out']    = (data['newbalanceOrig'] == 0).astype(int)

# One-hot encode transaction type
txn_type_encoded = pd.get_dummies(data['type'])
data = data.drop(columns=['type']).join(txn_type_encoded)

# Derive destination account category
data['dest_account_type'] = data['nameDest'].str[:1]
data = data.drop(columns=['nameOrig', 'nameDest'])

dest_encoded = pd.get_dummies(data['dest_account_type'])
data = data.drop(columns=['dest_account_type']).join(dest_encoded)

# Drop M column (merchant accounts never targeted by fraud here)
if 'M' in data.columns:
    data = data.drop(columns=['M'])

print('Feature list after engineering:')
for i, col in enumerate(data.columns, 1):
    print(f'  {i:>2}. {col}')

## Section F — Class Balancing via Undersampling

In [ ]:
fraud_records  = data[data['isFraud'] == 1]
normal_records = data[data['isFraud'] == 0]

SAMPLE_SIZE = 8000
fraud_sampled  = fraud_records.sample(n=SAMPLE_SIZE, random_state=99)
normal_sampled = normal_records.sample(n=SAMPLE_SIZE, random_state=99)

model_df = pd.concat([fraud_sampled, normal_sampled])
model_df = model_df.sample(frac=1, random_state=99).reset_index(drop=True)

print(f'Balanced dataset: {model_df.shape}')
print(model_df['isFraud'].value_counts())

## Section G — Min-Max Normalisation

In [ ]:
def apply_minmax(df, col):
    """Normalise column values to range [0, 1]."""
    v_min = df[col].min()
    v_max = df[col].max()
    if v_max > v_min:
        df[col] = (df[col] - v_min) / (v_max - v_min)
    else:
        df[col] = 0.0

cols_to_scale = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
                  'oldbalanceDest', 'newbalanceDest',
                  'txn_to_balance_ratio']

for c in cols_to_scale:
    apply_minmax(model_df, c)

print('Normalisation complete.')
model_df.describe().round(3)

## Section H — Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

predictors   = model_df.drop(columns=['isFraud'])
outcome      = model_df['isFraud']

X_tr, X_te, y_tr, y_te = train_test_split(
    predictors, outcome,
    test_size=0.2,
    random_state=99,
    stratify=outcome
)

print(f'Training   : {X_tr.shape[0]} samples')
print(f'Testing    : {X_te.shape[0]} samples')

## Section I — Ensemble Model: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay,
                              roc_auc_score, average_precision_score,
                              roc_curve, precision_recall_curve, classification_report)
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

forest_model = RandomForestClassifier(n_jobs=-1, random_state=99)
forest_model.fit(X_tr, y_tr)
forest_preds = forest_model.predict(X_te)

print('=== Random Forest — Default Parameters ===')
print(f'Accuracy  : {accuracy_score(y_te, forest_preds):.4f}')
print(f'Precision : {precision_score(y_te, forest_preds):.4f}')
print(f'Recall    : {recall_score(y_te, forest_preds):.4f}')
print(f'F1 Score  : {f1_score(y_te, forest_preds):.4f}')

In [ ]:
# Confusion matrix
cm_forest = confusion_matrix(y_te, forest_preds)
ConfusionMatrixDisplay(confusion_matrix=cm_forest,
                       display_labels=['Normal', 'Fraud']).plot(cmap='GnBu')
plt.title('Random Forest — Confusion Matrix (Default)')
plt.show()

In [ ]:
# Hyperparameter search (optimising for F1)
search_space = {'n_estimators': randint(60, 400), 'max_depth': randint(3, 18)}

hp_search = RandomizedSearchCV(
    RandomForestClassifier(n_jobs=-1, random_state=99),
    param_distributions=search_space,
    n_iter=10, cv=5,
    scoring='f1',
    random_state=99,
    n_jobs=-1
)
hp_search.fit(X_tr, y_tr)

optimised_forest = hp_search.best_estimator_
opt_preds = optimised_forest.predict(X_te)

print('Best params:', hp_search.best_params_)
print('\n=== Random Forest — Optimised ===')
print(f'Accuracy  : {accuracy_score(y_te, opt_preds):.4f}')
print(f'Precision : {precision_score(y_te, opt_preds):.4f}')
print(f'Recall    : {recall_score(y_te, opt_preds):.4f}')
print(f'F1 Score  : {f1_score(y_te, opt_preds):.4f}')

In [ ]:
# Feature importance chart
feat_imp = pd.Series(
    optimised_forest.feature_importances_,
    index=X_tr.columns
).sort_values(ascending=True).tail(12)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh', color='#4C72B0')
plt.title('Top 12 Feature Importances — Optimised Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## Section J — Baseline: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

linreg_clf = LogisticRegression(max_iter=1000, random_state=99)
linreg_clf.fit(X_tr, y_tr)
linreg_preds = linreg_clf.predict(X_te)
linreg_proba = linreg_clf.predict_proba(X_te)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_te, linreg_preds, target_names=['Normal', 'Fraud']))

In [ ]:
# ROC Curve
fpr_lr, tpr_lr, _ = roc_curve(y_te, linreg_proba)
auroc_val = roc_auc_score(y_te, linreg_proba)

plt.figure(figsize=(8, 5))
plt.plot(fpr_lr, tpr_lr, lw=2, color='#C44E52', label=f'LR (AUROC={auroc_val:.3f})')
plt.plot([0,1],[0,1], 'k--', alpha=0.5, label='No Skill')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Logistic Regression — ROC Curve')
plt.legend()
plt.show()

In [ ]:
# Precision-Recall Curve
prec_lr, rec_lr, _ = precision_recall_curve(y_te, linreg_proba)
auprc_val = average_precision_score(y_te, linreg_proba)

plt.figure(figsize=(8, 5))
plt.step(rec_lr, prec_lr, where='post', color='#C44E52',
         label=f'LR (AUPRC={auprc_val:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Logistic Regression — Precision-Recall Curve')
plt.legend()
plt.show()

## Section K — Gradient Boosting: XGBoost


In [ ]:
! pip install xgboost -q

In [ ]:
from xgboost import XGBClassifier

xgb_estimator = XGBClassifier(
    eval_metric='logloss',
    random_state=99,
    use_label_encoder=False,
    n_estimators=300
)

xgb_estimator.fit(
    X_tr, y_tr,
    eval_set=[(X_te, y_te)],
    early_stopping_rounds=10,
    verbose=False
)

xgb_preds = xgb_estimator.predict(X_te)
xgb_proba = xgb_estimator.predict_proba(X_te)[:, 1]

print('=== XGBoost Classifier ===')
print(classification_report(y_te, xgb_preds, target_names=['Normal', 'Fraud']))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_te, xgb_preds,
    display_labels=['Normal', 'Fraud'],
    cmap='OrRd'
)
plt.title('XGBoost — Confusion Matrix')
plt.show()

## Section L — ROC Curve Comparison Across All Models

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

forest_proba  = optimised_forest.predict_proba(X_te)[:, 1]

fig, ax = plt.subplots(figsize=(9, 6))

for model_name, proba in [('Logistic Regression', linreg_proba),
                            ('Random Forest',       forest_proba),
                            ('XGBoost',             xgb_proba)]:
    fpr, tpr, _ = roc_curve(y_te, proba)
    auc_val      = roc_auc_score(y_te, proba)
    ax.plot(fpr, tpr, lw=2, label=f'{model_name} (AUC={auc_val:.3f})')

ax.plot([0,1],[0,1], 'k--', alpha=0.4, label='No Skill')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — All Models')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Section M — Performance Summary

In [ ]:
summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest (Default)', 'Random Forest (Optimised)', 'XGBoost'],
    'Accuracy' : [
        accuracy_score(y_te, linreg_preds),
        accuracy_score(y_te, forest_preds),
        accuracy_score(y_te, opt_preds),
        accuracy_score(y_te, xgb_preds)
    ],
    'Precision': [
        precision_score(y_te, linreg_preds),
        precision_score(y_te, forest_preds),
        precision_score(y_te, opt_preds),
        precision_score(y_te, xgb_preds)
    ],
    'Recall'   : [
        recall_score(y_te, linreg_preds),
        recall_score(y_te, forest_preds),
        recall_score(y_te, opt_preds),
        recall_score(y_te, xgb_preds)
    ],
    'F1 Score' : [
        f1_score(y_te, linreg_preds),
        f1_score(y_te, forest_preds),
        f1_score(y_te, opt_preds),
        f1_score(y_te, xgb_preds)
    ],
}).set_index('Model').round(4)

summary

In [ ]:
# Grouped bar comparison
summary[['Precision','Recall','F1 Score']].plot(
    kind='bar', figsize=(12, 5), colormap='tab10', edgecolor='white'
)
plt.title('Comparative Model Performance — Precision / Recall / F1')
plt.ylabel('Score')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1.05)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()